# Engineer departure-model features

Create the deterministic Appendix B features available before pushback. The output retains source and audit columns, but later Model 1A code must construct predictors from the explicit candidate allowlist in `feature_engineering.py`. No learned preprocessing is performed here.

In [ ]:
YEAR = 2019

AIRPORT = "JFK"

## Configure the merged input and feature output

The notebook works from either the project root or the `notebooks` directory. `AIRPORT` is normalized to uppercase, and the input must already contain departures from that airport.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

if (Path.cwd() / "data").is_dir() and (Path.cwd() / "notebooks").is_dir():
    PROJECT_ROOT = Path.cwd()
elif Path.cwd().name == "notebooks" and (Path.cwd().parent / "data").is_dir():
    PROJECT_ROOT = Path.cwd().parent
else:
    raise FileNotFoundError(
        "Start this notebook from the capstone project root or its notebooks directory"
    )

NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))

from feature_engineering import (
    MODEL_TARGETS,
    PRE_ENGINEERED_FEATURES,
    add_pre_features,
    model_feature_candidates,
    validate_engineered_features,
)

AIRPORT = str(AIRPORT).strip().upper()
YEAR = int(YEAR)
INPUT_FILE = PROJECT_ROOT / f"data/merged/{AIRPORT}_{YEAR}_departures.csv"
OUTPUT_FILE = PROJECT_ROOT / f"data/features/{AIRPORT}_{YEAR}_departures.csv"

print(pd.Series({"input": str(INPUT_FILE), "output": str(OUTPUT_FILE)}))

## Load and validate the departure population

The notebook validates rather than silently filtering the merged dataset. This prevents an incorrectly assembled airport or year from producing a plausible-looking feature file.

In [ ]:
if not INPUT_FILE.is_file():
    raise FileNotFoundError(f"Merged departure file does not exist: {INPUT_FILE}")

source = pd.read_csv(INPUT_FILE, low_memory=False)
required_scope_columns = {"Year", "Origin", MODEL_TARGETS["1A"]}
missing_scope_columns = required_scope_columns - set(source.columns)
if missing_scope_columns:
    raise KeyError(f"Merged departure data is missing: {sorted(missing_scope_columns)}")

origin = source["Origin"].astype("string").str.strip().str.upper()
source_year = pd.to_numeric(source["Year"], errors="coerce")
if not origin.eq(AIRPORT).all():
    raise ValueError(f"Departure input contains origins other than {AIRPORT}")
if not source_year.eq(YEAR).all():
    raise ValueError(f"Departure input contains years other than {YEAR}")

target = pd.to_numeric(source[MODEL_TARGETS["1A"]], errors="coerce")
if target.isna().any() or not target.isin([0, 1]).all():
    raise ValueError("DepDel15 must be complete and binary")

print(pd.Series({"rows": len(source), "columns": len(source.columns), "delayed": int(target.sum())}))

## Add the pre-pushback features

All transformations are fixed formulas. Missing source values propagate into dependent features, including ASPM three-hour aggregates at annual boundaries.

In [ ]:
features = add_pre_features(source)
if len(features) != len(source):
    raise ValueError("Feature engineering changed the departure row count")
if not features[source.columns].equals(source):
    raise ValueError("Feature engineering changed one or more source columns")

feature_validation = validate_engineered_features(
    features, PRE_ENGINEERED_FEATURES
)
model_1a_candidates = model_feature_candidates("1A", features.columns)
if MODEL_TARGETS["1A"] in model_1a_candidates:
    raise ValueError("Model 1A target leaked into its feature allowlist")

print(pd.Series({
    "engineered features": len(PRE_ENGINEERED_FEATURES),
    "Model 1A candidate columns": len(model_1a_candidates),
    "rows with missing engineered values": int(
        features[PRE_ENGINEERED_FEATURES].isna().any(axis=1).sum()
    ),
}))

In [ ]:
feature_validation

## Save the departure feature dataset

The CSV retains the audit columns and target. Later model code must import the Model 1A allowlist rather than treating every non-target column as a predictor.

In [ ]:
OUTPUT_FILE.parent.mkdir(parents=True, exist_ok=True)
features.to_csv(OUTPUT_FILE, index=False)

summary = pd.Series({
    "airport": AIRPORT,
    "year": YEAR,
    "rows": len(features),
    "columns": len(features.columns),
    "target": MODEL_TARGETS["1A"],
    "output": str(OUTPUT_FILE),
}, name="departure feature summary")
print(f"Saved {len(features):,} rows to {OUTPUT_FILE}")
summary